In [37]:
import pandas as pd
import psycopg2

In [38]:
# Database connection parameters
DB_NAME = 'bank_reviews'
DB_USER = 'postgres'
DB_PASSWORD = '1024'
DB_HOST = 'localhost'  
DB_PORT = '5432' 

In [39]:
df = pd.read_csv("bank_reviews_with_sentiment_and_themes.csv")

In [40]:
df[:1]

,review,rating,date,bank,source,sentiment,identified_theme
0,what is this app problem???,1,2025-06-05,CBE,Google Play Store,NEGATIVE,Other


In [41]:
try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    cursor = conn.cursor()
    
    # Insert data into the reviews table
    for index, row in df.iterrows():
        cursor.execute("""
            INSERT INTO reviews (review, rating, date, bank, source, sentiment, identified_theme)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (row['review'], row['rating'], row['date'], row['bank'], row['source'], row['sentiment'], row['identified_theme']))

    conn.commit()
    print("Data inserted successfully.")

except Exception as e:
    print("Error:", e)


Data inserted successfully.


In [18]:
def insert_banks():
    banks = ['CBE', 'BOA', 'Dashen']
    
    try:
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            host=DB_HOST,
            port=DB_PORT
        )
        cursor = conn.cursor()

        for bank in banks:
            cursor.execute("INSERT INTO banks (name) VALUES (%s);", (bank,))
        
        conn.commit()
        print("Banks inserted successfully.")

    except Exception as e:
        print("Error:", e)
        if conn:
            conn.rollback()  # Rollback the transaction on error

    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

if __name__ == "__main__":

    insert_banks()   # Then insert the banks

Banks inserted successfully.


In [42]:
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

In [44]:
def read_data():
    try:
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            host=DB_HOST,
            port=DB_PORT
        )
        query = """
            SELECT r.review, r.rating, r.date, r.bank, r.source, r.sentiment, r.identified_theme, b.id
            FROM reviews r
            JOIN banks b ON r.bank = b.bank;
        """
        df = pd.read_sql(query, conn)
        return df
    except Exception as e:
        print("Error:", e)
    finally:
        if conn:
            conn.close()

# Load the data
data = read_data()

/tmp/ipykernel_25487/2194743599.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [45]:
data

,review,rating,date,bank,source,sentiment,identified_theme,id
0,what is this app problem???,1,2025-06-05,CBE,Google Play Store,NEGATIVE,Other,1
1,the app is proactive and a good connections.,5,2025-06-05,CBE,Google Play Store,POSITIVE,Other,1
2,I cannot send to cbebirr app. through this app.,3,2025-06-05,CBE,Google Play Store,NEGATIVE,Other,1
3,good,4,2025-06-05,CBE,Google Play Store,POSITIVE,Other,1
4,not functional,1,2025-06-05,CBE,Google Play Store,NEGATIVE,Other,1
...,...,...,...,...,...,...,...,...
6664,Faster and userfriendly,5,2025-01-14,Dashen,Google Play Store,POSITIVE,Other,3
6665,"Waw Great and innovated,user friendly, always ...",5,2025-01-13,Dashen,Google Play Store,POSITIVE,Other,3
6666,It's Best waww 🙏,5,2025-01-13,Dashen,Google Play Store,POSITIVE,Other,3
6667,Always one step ahead,5,2025-01-13,Dashen,Google Play Store,POSITIVE,Other,3
